# Demo: la API de producción en vivo

Corrida de ejemplo del flujo completo contra la API real (FastAPI + worker + Redis + Gemini + LangSmith):

1. Validaciones de la API: startup inexistente (404) y aprobación sin clave (401).
2. **Caso no crítico** (VozActiva): un descarte claro se emite solo, sin pausa humana.
3. **Caso crítico** (Lupa Code): el grafo se detiene con `interrupt()` y el job queda en `awaiting_approval` con el pedido de aprobación (nivel de riesgo y motivos).
4. **Otra vuelta**: el comité pide revisar un punto; el supervisor lo delega y el análisis vuelve a la pausa.
5. **Aprobación**: el worker reanuda el hilo desde el checkpoint de Redis y el job termina en `done`.

Requisitos: la API (`uvicorn app.main:app --port 8000`) y el worker (`python -m app.worker`) corriendo. Usa unas 30 llamadas al LLM.

> **Nota de reproducibilidad.** Este notebook guarda las **salidas de una ejecución real** del 2026-11-18 contra la API (FastAPI + worker + Redis Cloud 8.8 + Gemini 3.5 Flash-Lite + LangSmith). Se puede leer tal como está, sin ejecutar nada.
>
> Para volver a ejecutarlo hace falta:
> 1. la API (`uvicorn app.main:app --port 8000`) y el worker (`python -m app.worker`) corriendo;
> 2. un Redis 8 o Redis Cloud en `REDIS_URL`;
> 3. `GEMINI_API_KEY` y `API_KEY_APROBACION` en el `.env` (ver el README).
>
> Para validar la lógica sin servicios externos ni credenciales: `pytest` (163 tests).

In [1]:
import json, os, time
import httpx
from dotenv import load_dotenv

load_dotenv()
API = httpx.Client(base_url='http://localhost:8000', timeout=30)
CLAVE = {'X-API-Key': os.environ['API_KEY_APROBACION']}

def esperar(job_id, estados, cada=3, limite=900):
    fin = time.monotonic() + limite
    while time.monotonic() < fin:
        job = API.get(f'/tasks/{job_id}').json()
        if job['estado'] in estados | {'done', 'rejected', 'failed'}:
            return job
        time.sleep(cada)
    raise TimeoutError(job_id)

def eventos(job, desde=0):
    for e in job['eventos'][desde:]:
        print(' ', e)

salud = API.get('/health').json()
print(json.dumps({k: salud[k] for k in ('estado', 'redis', 'observabilidad')}, ensure_ascii=False, indent=1))

{
 "estado": "ok",
 "redis": "ok",
 "observabilidad": {
  "plataforma": "langsmith",
  "activa": true,
  "proyecto": "due-diligence-pe7-produccion"
 }
}


## 1. Validaciones de la API

In [2]:
r = API.post('/tasks', json={'startup': 'inexistente'})
print(r.status_code, r.json()['detail'])
r = API.post('/tasks/cualquiera/approve', json={'decision': 'aprobar'})
print(r.status_code, r.json()['detail'])

404 No existe la startup 'inexistente'. Opciones: anamnesis, aulaviva, contralia, lupacode, vozactiva.
401 Header X-API-Key ausente o inválido.


## 2. Caso no crítico: VozActiva

El `POST` responde en milisegundos con el `job_id`; el análisis corre en el worker.

In [3]:
t0 = time.perf_counter()
r = API.post('/tasks', json={'startup': 'vozactiva'})
print(r.status_code, f'{(time.perf_counter() - t0) * 1000:.0f} ms', r.json())
voz = esperar(r.json()['job_id'], {'awaiting_approval'})
if voz['estado'] == 'awaiting_approval':  # score cerca de un umbral: el triage lo marcó crítico
    print('El triage lo marcó crítico:', voz['aprobacion']['motivos_pausa'])
    API.post(f"/tasks/{voz['job_id']}/approve", json={'decision': 'aprobar'}, headers=CLAVE)
    voz = esperar(voz['job_id'], set())
eventos(voz)
print('\nEstado:', voz['estado'], '·', voz['recomendacion'], '·', voz['decision_final'])
print('Criticidad:', json.dumps(voz['criticidad'], ensure_ascii=False, indent=1))

202 151 ms {'job_id': '62b2e7cd1d894b63941dab5ab4793ddb', 'estado': 'pending', 'startup': 'vozactiva', 'links': {'estado': '/tasks/62b2e7cd1d894b63941dab5ab4793ddb', 'informe': '/tasks/62b2e7cd1d894b63941dab5ab4793ddb/informe', 'eventos': '/tasks/62b2e7cd1d894b63941dab5ab4793ddb/eventos', 'aprobar': '/tasks/62b2e7cd1d894b63941dab5ab4793ddb/approve'}}


  ▶ comienza la due diligence de VozActiva
  [paso 1] supervisor → investigador: Resolvé este punto de la rúbrica: El investigador todavía no entregó datos.
  motivo: Orden natural de la rúbrica: El investigador todavía no entregó datos. [guarda: decisión determinística, sin LLM]
  investigador: v1: 21 métricas ancladas, 3 hallazgos, 3 registros rechazados
  [paso 2] supervisor → analista: Resolvé este punto de la rúbrica: El analista todavía no calculó.
  motivo: Orden natural de la rúbrica: El analista todavía no calculó. [guarda: decisión determinística, sin LLM]
  analista: v1: costo LLM USD 16.800/mes (18,7 % del MRR) · margen 41,9 % · LTV/CAC 0,93 · payback 16,6 m · runway 7,4 m · score 41,0 (no avanzar); 0 llamadas rechazadas por falta de respaldo
  [paso 3] supervisor → critico: Resolvé este punto de la rúbrica: El crítico todavía no auditó el análisis.
  motivo: Orden natural de la rúbrica: El crítico todavía no auditó el análisis. [guarda: decisión determinística, sin LLM]
  

## 3. Caso crítico: Lupa Code se detiene en la aprobación humana

In [4]:
lupa_id = API.post('/tasks', json={'startup': 'lupacode'}).json()['job_id']
lupa = esperar(lupa_id, {'awaiting_approval'})
eventos(lupa)
print('\nEstado:', lupa['estado'])
print('Pedido de aprobación (payload del interrupt):')
print(json.dumps(lupa['aprobacion'], ensure_ascii=False, indent=1))

  ▶ comienza la due diligence de Lupa Code
  [paso 1] supervisor → investigador: Resolvé este punto de la rúbrica: El investigador todavía no entregó datos.
  motivo: Orden natural de la rúbrica: El investigador todavía no entregó datos. [guarda: decisión determinística, sin LLM]
  investigador: v1: 21 métricas ancladas, 3 hallazgos, 3 registros rechazados
  [paso 2] supervisor → analista: Resolvé este punto de la rúbrica: El analista todavía no calculó.
  motivo: Orden natural de la rúbrica: El analista todavía no calculó. [guarda: decisión determinística, sin LLM]
  analista: v1: costo LLM USD 17.250/mes (28,8 % del MRR) · margen 60,4 % · LTV/CAC 3,87 · payback 10,3 m · runway 20,0 m · score 89,0 (avanzar a comité); 0 llamadas rechazadas por falta de respaldo
  [paso 3] supervisor → critico: Resolvé este punto de la rúbrica: El crítico todavía no auditó el análisis.
  motivo: Orden natural de la rúbrica: El crítico todavía no auditó el análisis. [guarda: decisión determinística, sin 

## 4. El comité pide otra vuelta

El comentario es obligatorio. El supervisor (LLM) decide qué especialista atiende el pedido.

In [5]:
vistos = len(lupa['eventos'])
pedido = {'decision': 'otra_vuelta', 'aprobador': 'Ana (comité)',
          'comentario': 'Revisá cuánto cae el margen cuando vencen los créditos promocionales del LLM.'}
print(API.post(f'/tasks/{lupa_id}/approve', json=pedido, headers=CLAVE).json())
lupa = esperar(lupa_id, {'awaiting_approval'})
eventos(lupa, vistos)
print('\nSegunda pausa:', lupa['estado'], '· rondas pedidas:', lupa['aprobacion']['rondas_pedidas'],
      '· acciones:', lupa['aprobacion']['acciones'])

{'job_id': 'b9dce131a702498eb93febf0c07a8568', 'estado': 'pending', 'decision': 'otra_vuelta', 'mensaje': 'Decisión registrada: el worker retoma el análisis desde el checkpoint.', 'links': {'estado': '/tasks/b9dce131a702498eb93febf0c07a8568', 'informe': '/tasks/b9dce131a702498eb93febf0c07a8568/informe', 'eventos': '/tasks/b9dce131a702498eb93febf0c07a8568/eventos', 'aprobar': '/tasks/b9dce131a702498eb93febf0c07a8568/approve'}}


  ▶ se reanuda con la decisión humana: otra_vuelta
  aprobacion: Ana (comité): otra_vuelta — Revisá cuánto cae el margen cuando vencen los créditos promocionales del LLM.
  [paso 5] supervisor → analista: Calculá el impacto en el margen bruto y los unit economics de Lupa Code asumiendo que vencen los créditos promocionales de los proveedores de LLM, incrementando el costo de inferencia según las tarifas públicas (on-deman…
  motivo: Aunque la rúbrica está completa, el pedido del comité sobre el impacto de los créditos promocionales en el margen sigue pendiente. Debo enviar al analista para recalcular los unit economics bajo este escenario de vencimiento de créditos.
  analista: v2: costo LLM USD 17.250/mes (28,8 % del MRR) · margen 60,4 % · LTV/CAC 3,87 · payback 10,3 m · runway 20,0 m · score 89,0 (avanzar a comité); 1 llamadas rechazadas por falta de respaldo
  [paso 6] supervisor → critico: Resolvé este punto de la rúbrica: La crítica v1 auditó el análisis v1, pero el vigente es el 

## 5. Aprobación: el worker reanuda el hilo desde el checkpoint

In [6]:
vistos = len(lupa['eventos'])
print(API.post(f'/tasks/{lupa_id}/approve', json={'decision': 'aprobar', 'aprobador': 'Ana (comité)'}, headers=CLAVE).json())
lupa = esperar(lupa_id, set())
eventos(lupa, vistos)
print('\nEstado:', lupa['estado'], '·', lupa['recomendacion'], '·', lupa['decision_final'])
informe = API.get(f'/tasks/{lupa_id}/informe').text
print(informe[informe.index('## Decisión del comité'):])

{'job_id': 'b9dce131a702498eb93febf0c07a8568', 'estado': 'pending', 'decision': 'aprobar', 'mensaje': 'Decisión registrada: el worker retoma el análisis desde el checkpoint.', 'links': {'estado': '/tasks/b9dce131a702498eb93febf0c07a8568', 'informe': '/tasks/b9dce131a702498eb93febf0c07a8568/informe', 'eventos': '/tasks/b9dce131a702498eb93febf0c07a8568/eventos', 'aprobar': '/tasks/b9dce131a702498eb93febf0c07a8568/approve'}}


  ▶ se reanuda con la decisión humana: aprobar
  aprobacion: Ana (comité): aprobar
  cierre: recomendación PROFUNDIZAR aprobada por el comité
  ■ PROFUNDIZAR — aprobada por el comité

Estado: done · PROFUNDIZAR · APROBADA


## Decisión del comité (human-in-the-loop)
**APROBADA**.
- Pausa 1 (2026-11-18T23:27:35) — Ana (comité) eligió **otra_vuelta**: Revisá cuánto cae el margen cuando vencen los créditos promocionales del LLM.
- Pausa 2 (2026-11-18T23:29:03) — Ana (comité) eligió **aprobar**



## Uso del LLM y costo propio por ejecución

In [7]:
for job in (voz, lupa):
    print(f"{job['startup']}: {job['llamadas_llm']} llamadas · USD {job['costo_usd']:.5f} · {job['ejecuciones']} ejecuciones")
    for e in job['ejecuciones_detalle']:
        print(f"   {e['tipo']:<9} {e['latencia_s']:>6.1f} s · {e['uso_llm']['total']['llamadas']:>2} llamadas · USD {e['costo']['total_usd']:.5f}")

vozactiva: 12 llamadas · USD 0.02213 · 1 ejecuciones
   iniciar    158.4 s · 12 llamadas · USD 0.02213
lupacode: 26 llamadas · USD 0.03523 · 3 ejecuciones
   iniciar    156.3 s · 13 llamadas · USD 0.02161
   reanudar    85.3 s · 13 llamadas · USD 0.01362
   reanudar     4.7 s ·  0 llamadas · USD 0.00000
